In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))
from anchor import data_dir

pd.set_option("display.max_columns", 200)

In [ ]:
ANONYMIZATION_PATH = os.path.join(data_dir(), "personai")

IN_SURVEY = os.path.join(ANONYMIZATION_PATH, "Onboarding survey_all participants_cleaned_non_anon.xlsx")
IN_SCORES = os.path.join(ANONYMIZATION_PATH, "Scores_non_anon.csv")
IN_FINAL_SURVEY = os.path.join(ANONYMIZATION_PATH, "final_onboarding_survey_non_anon.csv")
IN_ONBOARDING = os.path.join(ANONYMIZATION_PATH,"onboarding_survey_non_anon.csv")


OUT_SURVEY = os.path.join(ANONYMIZATION_PATH, "Onboarding survey_all participants_cleaned.xlsx")
OUT_SCORES = os.path.join(ANONYMIZATION_PATH, "Scores.csv")
OUT_FINAL_SURVEY = os.path.join(ANONYMIZATION_PATH, "final_onboarding_survey.csv")#
OUT_ONBOARDING = os.path.join(ANONYMIZATION_PATH,"onboarding_survey.csv")

In [ ]:
df_survey = pd.read_excel(IN_SURVEY)
df_scores = pd.read_csv(IN_SCORES, low_memory=False)
df_fin_survey = pd.read_csv(IN_FINAL_SURVEY, low_memory=False)
df_onboarding = pd.read_csv(IN_ONBOARDING,low_memory = False)


In [ ]:
# def anonymize_columns(df,cols):
#     df[cols] = 'x'
#     return df

def anonymize_columns(df, cols):
    norm = lambda s: " ".join(str(s).split())
    wanted = {norm(c) for c in cols}
    hit = [c for c in df.columns if norm(c) in wanted]
    df[hit] = 'x'
    return df

to_anon = ['Sind Sie aktuell schwanger?', 'Alter:',
       'Höchster Bildungsabschluss:\r\n',
       'Haben Sie mehrere Jahre (5+) außerhalb von Österreich gelebt?',
       'In welchem Land haben Sie gelebt und seit wann sind Sie (wieder) in Österreich?',
       'Wohnort (PLZ, Ort):\r\n', 'Art der Wohnimmobilie:\r\n',
       'Baualter der Immobilie',
       'Bitte geben Sie die Bauperiode der Immobilie an, sofern Sie diese kennen:\r\n',
       'Klimatisierungsmöglichkeit:\r\n', 'Lüftung und Lüftungsverhalten:',
       'Heizungssystem:',
       'Wurde die Immobilie in den letzten 20 Jahren saniert?\r\n',
       'Falls ja, wann?\r\n', 'Umfang der Sanierung',
       'Wie würden Sie das durchschnittliche Raumklima in Ihrem Zuhause einschätzen?\r\nUnter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw. ',
       'Berufliche Tätigkeit (vorwiegend):\r\n',
       'Wie viele Stunden arbeiten Sie durchschnittlich pro Woche?',
       'Arbeitsort (PLZ, Ort):Falls Sie mehrere Arbeitsorte haben, nennen Sie bitte den, an dem Sie die meiste Zeit verbringen.',
       'Welche Arbeitsorte nutzen Sie regelmäßig im Rahmen Ihrer Arbeit? (Mehrauswahl möglich)',
       'An welchen Wochentagen arbeiten Sie normalerweise?',
       'An welchen Wochentagen arbeiten Sie normalerweise im Büro?',
       'An welchen Wochentagen arbeiten Sie normalerweise im Home-Office?',
       'An welchen Wochentagen arbeiten Sie normalerweise remote (zB in Cafés, in der Bibliothek)?',
       'Geben Sie bitte die Räumlichkeiten in Ihrem Büro an:',
       'Klimatisierungsmöglichkeit im Büro:\r\n',
       'Lüftung und Lüftungsverhalten im Büro:', 'Heizungssystem im Büro:',
       'Wie würden  Sie das durchschnittliche Raumklima in Ihrem Büro einschätzen?\r\nUnter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw. ']

df_survey = anonymize_columns(df_survey,to_anon)
df_scores = anonymize_columns(df_scores,to_anon)
df_fin_survey = anonymize_columns(df_fin_survey,to_anon)
df_onboarding = anonymize_columns(df_onboarding,to_anon)


In [ ]:
df_fin_survey.to_csv(OUT_FINAL_SURVEY, index=False)
df_scores.to_csv(OUT_SCORES, index=False)
df_survey.to_excel(OUT_SURVEY,index=False)
df_onboarding.to_csv(OUT_ONBOARDING,index = False)

In [ ]:

list_dir = os.listdir(os.path.join(data_dir(), "personai", "Person_AI_Data", "Data"))
for f in list_dir:
    if f[:2] == "id":
        
        RAW_ROOT = os.path.join(data_dir(), "personai", "Person_AI_Data", "Data", f)

        # columns that make_hr_aggregated.py reads; all other columns are set to 'x'
        NEEDED = {
            "Cozie": ["index", "ts_heart_rate", "ws_heart_rate","id_participant"],
            "Oura": ["timestamp", "bpm","participant_id"],
        }

        for sensor, needed in NEEDED.items():
            folder = os.path.join(RAW_ROOT, sensor)
            for name in os.listdir(folder):
                if name.endswith("_anon.csv") or not name.startswith(("sample", "heartrate")):
                    continue
                df = pd.read_csv(os.path.join(folder, name), dtype=str)
                print(df.head())
                df.loc[:, ~df.columns.isin(needed)] = "x"
                print(df.head())
                df.to_csv(os.path.join(folder, name.replace(".csv", "_anon.csv")), index=False)

In [ ]:
df